In [ ]:
import setuptools
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from pycaret.classification import (
    compare_models,
    evaluate_model,
    plot_model,
    setup,
    tune_model,
)

RuntimeError: ('Pycaret only supports python 3.9, 3.10, 3.11. Your actual Python version: ', sys.version_info(major=3, minor=12, micro=6, releaselevel='final', serial=0), 'Please DOWNGRADE your Python version.')

: 

In [52]:
alarmes = pd.read_csv('../../data/Alarmes/alarmes_trata_limpos_consolidado.csv')
zm01 = pd.read_excel('../../data/Corretiva Emergencial/processados/ZM01_limpa.xlsx', sheet_name='ZM01_limpa')
zm02 = pd.read_excel('../../data/Corretiva Programada/processados/ZM02_limpa.xlsx', sheet_name='ZM02_limpa')
zm03 = pd.read_csv('../../data/Preventiva/preventiva_zm03_limpa.csv')

print(alarmes.shape, zm01.shape, zm02.shape, zm03.shape)

(742506, 15) (10543, 27) (1163, 27) (569, 19)


In [53]:
alarmes.head(5)

numeroAlarmesEstacao = alarmes['Estação'].value_counts()

## Pré Processamento

### Transformando em Datetime

In [96]:
zm01["T"] = pd.to_datetime(zm01["inicio_avaria_datetime"], errors="coerce")
zm01['fim_avaria_datetime'] = pd.to_datetime(zm01['fim_avaria_datetime'], errors='coerce')
alarmes['Horário do Alarme'] = pd.to_datetime(alarmes['Horário do Alarme'], errors='coerce')
alarmes['dt_alarme'] = pd.to_datetime(alarmes['dt_alarme'], errors='coerce')
alarmes['dt_normalizacao'] = pd.to_datetime(alarmes['dt_normalizacao'], errors='coerce')
alarmes.head(5)

,TAG,Horário do Alarme,Estação,Porta,Alarme,Horário da Normalizacao,Tempo de Normalização,tempo_normalizacao_seg,dt_alarme,dt_normalizacao,duracao_calculada_seg,estacao_extraida,plataforma_extraida,porta_extraida,tipo_evento_extraido
0,PSD_BUT_BUT1_PDM18_OBS,2023-10-17 21:44:30.900,BUT,18.0,OBS,2023-10-23 01:17:58.893,"5 dias, 03:33:28",444808,2023-10-17 21:44:30.900,2023-10-23 01:17:58.893,444807.993,BUT,BUT1,18.0,OBS
1,PSD_BUT_BUT1_PDM18_OBS,2023-10-17 21:44:30.900,BUT,18.0,OBS,2023-10-21 14:14:04.483,"3 dias, 16:29:34",318574,2023-10-17 21:44:30.900,2023-10-21 14:14:04.483,318573.583,BUT,BUT1,18.0,OBS
2,PSD_BUT_BUT1_PDM18_OBS,2023-10-17 21:44:30.900,BUT,18.0,OBS,2023-10-21 12:08:57.967,"3 dias, 14:24:27",311067,2023-10-17 21:44:30.900,2023-10-21 12:08:57.967,311067.067,BUT,BUT1,18.0,OBS
3,PSD_BUT_BUT1_PDM18_OBS,2023-10-17 21:44:30.900,BUT,18.0,OBS,2023-10-21 11:39:45.640,"3 dias, 13:55:15",309315,2023-10-17 21:44:30.900,2023-10-21 11:39:45.640,309314.740,BUT,BUT1,18.0,OBS
4,PSD_P_BUT1_PDM18_CA,2023-10-17 21:44:30.900,BUT,18.0,CA,2023-10-21 11:39:45.640,"3 dias, 13:55:15",309315,2023-10-17 21:44:30.900,2023-10-21 11:39:45.640,309314.740,BUT,BUT1,18.0,CA


In [98]:
# 1. Calcula a duração em segundos (caso ainda não tenha essa coluna)
alarmes["duracao_segundos"] = (
    alarmes["dt_normalizacao"] - alarmes["dt_alarme"]
).dt.total_seconds()

# 2. Identifica os alarmes de OBS com menos de 30s
obs_curto = alarmes["Alarme"].astype(str).str.contains(
    "OBS", case=False, na=False
) & (alarmes["duracao_segundos"] < 30)

# 3. Mantém apenas os alarmes válidos na tabela
alarmes = alarmes[~obs_curto].reset_index(drop=True)

In [99]:
# Ordena os DataFrames por estação e tempo
zm01 = zm01.sort_values(["estacao", "T"]).reset_index(drop=True)
alarmes = (
    alarmes.sort_values(["Estação", "dt_alarme"])
    .dropna(subset=["dt_alarme"])
    .reset_index(drop=True)
)

### Features

In [100]:
# Feature - Nome da estação
zm01['estacao'] = zm01['local_de_instalacao'].str.extract(r'PSD-([A-Z0-9]+)')

In [101]:
# Feature - Numero de Alarmes por Estação
zm01['numero_alarmes'] = zm01['estacao'].map(numeroAlarmesEstacao).fillna(0).astype(int)

In [102]:
# Feature - Alarmes Antes Manutencao

datas_alarmes_ordenadas = (
    alarmes["Horário do Alarme"].dropna().sort_values().values
)

total_alarmes = len(datas_alarmes_ordenadas)

zm01["alarmes_antes_manu"] = np.searchsorted(
    datas_alarmes_ordenadas, zm01["fim_avaria_datetime"].values, side="left"
)
zm01.loc[zm01["fim_avaria_datetime"].isna(), "alarmes_antes_manu"] = 0

In [103]:
# 1. Feature: dias_desde_ultima_manutencao
zm01["ultima_manu_fim"] = zm01.groupby("estacao")["fim_avaria_datetime"].shift(1)
zm01["dias_desde_ultima_manutencao"] = (zm01["T"] - zm01["ultima_manu_fim"]).dt.total_seconds() / 86400.0
zm01["dias_desde_ultima_manutencao"] = zm01["dias_desde_ultima_manutencao"].fillna(-1)

In [104]:
def calcular_janelas(df_zm, df_alt):
    cnt_24h = np.zeros(len(df_zm), dtype=int)
    cnt_7d = np.zeros(len(df_zm), dtype=int)

    for estacao, grp_zm in df_zm.groupby("estacao"):
        grp_alt = df_alt[df_alt["Estação"] == estacao]
        if grp_alt.empty:
            continue

        # Converte explicitamente para array nativo datetime64[ns] do NumPy (Evita o TypeError)
        times_alt = grp_alt["dt_alarme"].to_numpy(dtype="datetime64[ns]")
        times_zm = grp_zm["T"].to_numpy(dtype="datetime64[ns]")

        idx_t = np.searchsorted(times_alt, times_zm, side="right")
        idx_24h = np.searchsorted(
            times_alt, times_zm - np.timedelta64(24, "h"), side="left"
        )
        idx_7d = np.searchsorted(
            times_alt, times_zm - np.timedelta64(7, "D"), side="left"
        )

        cnt_24h[grp_zm.index] = idx_t - idx_24h
        cnt_7d[grp_zm.index] = idx_t - idx_7d

    return cnt_24h, cnt_7d

In [105]:
zm01["alarmes_ultimas_24h"], zm01["alarmes_ultimos_7d"] = calcular_janelas(zm01, alarmes)

media_diaria_7d = zm01["alarmes_ultimos_7d"] / 7.0
zm01["tendencia_alarmes"] = np.where(media_diaria_7d > 0, zm01["alarmes_ultimas_24h"] / media_diaria_7d, 0.0)

zm01['tendencia_alarmes'].value_counts()

tendencia_alarmes
0.000000    5069
7.000000      67
1.995098      23
0.052994      23
0.921374      22
            ... 
0.478632       1
0.663793       1
4.693798       1
4.569444       1
1.857516       1
Name: count, Length: 4551, dtype: int64

## Transformações de Variáveis Categóricas

In [107]:
zm01.dtypes

linha_origem                                     int64
tipo_de_nota                                       str
nota                                             int64
ordem                                          float64
perfil_do_catalogo                                 str
descricao                                          str
local_de_instalacao                                str
centro_de_trabalho_responsavel                     str
inicio_avaria_datetime                  datetime64[us]
fim_avaria_datetime                     datetime64[us]
duracao_avaria_horas                           float64
analise_datetime                        datetime64[us]
encerramento_datetime                   datetime64[us]
indisponivel_flag                                 bool
falha_flag                                        bool
analisada_flag                                    bool
ordem_ausente_flag                                bool
ordem_repetida_flag                               bool
comentario

### Label Encoder

In [114]:
colunas_categoricas = ["estacao", "perfil_do_catalogo", "defeito"]

label_encoders = {}

for col in colunas_categoricas:
    le = LabelEncoder()
    zm01[col] = le.fit_transform(zm01[col].astype(str))
    label_encoders[col] = le

zm01.head(5)

,linha_origem,tipo_de_nota,nota,ordem,perfil_do_catalogo,descricao,local_de_instalacao,centro_de_trabalho_responsavel,inicio_avaria_datetime,fim_avaria_datetime,...,estacao,numero_alarmes,alarmes_antes_manu,alarmes_depois_manu,ultima_manu_fim,T,dias_desde_ultima_manutencao,alarmes_ultimas_24h,alarmes_ultimos_7d,tendencia_alarmes
0,683,Z1,13036932,52078677.0,0,PDM 18 - Falha de fechamento,SP04-6-PSD-BUT-PLT2-PP18PDM,SP04IBUT,2023-01-03 04:18:00,2023-01-03 04:48:16,...,0,91805,0,742506,NaT,2023-01-03 04:18:00,-1.000000,0,0,0.0
1,533,Z1,13038801,52080470.0,0,PDM 07 Falha na abertura,SP04-6-PSD-BUT-PLT2-PP07PDM,SP04IBUT,2023-01-04 06:30:00,2023-01-04 07:00:00,...,0,91805,0,742506,2023-01-03 04:48:16,2023-01-04 06:30:00,1.070648,0,0,0.0
2,615,Z1,13045687,52086918.0,0,PDM11 - Falha no fechamento,SP04-6-PSD-BUT-PLT2-PP11PDM,SP04IBUT,2023-01-08 10:25:00,2023-01-08 15:00:13,...,0,91805,0,742506,2023-01-04 07:00:00,2023-01-08 10:25:00,4.142361,0,0,0.0
3,115,Z1,13052087,52093262.0,0,PDM 09 - Falha de fechamento,SP04-6-PSD-BUT-PLT1-PP09PDM,SP04IBUT,2023-01-11 18:18:00,2023-01-12 04:47:05,...,0,91805,0,742506,2023-01-08 15:00:13,2023-01-11 18:18:00,3.137350,0,0,0.0
4,750,Z1,13058808,52099639.0,0,PDM 24 Falha no fechamento,SP04-6-PSD-BUT-PLT2-PP24PDM,SP04IBUT,2023-01-15 14:23:00,2023-01-15 15:20:00,...,0,91805,0,742506,2023-01-12 04:47:05,2023-01-15 14:23:00,3.399942,0,0,0.0


## Separando Dados de Teste e Treino

In [115]:
zm01 = zm01.sort_values(["estacao", "T"]).reset_index(drop=True)

zm01["proxima_avaria_datetime"] = zm01.groupby("estacao")["T"].shift(-1)

zm01["dias_ate_proxima_falha"] = (
    zm01["proxima_avaria_datetime"] - zm01["T"]
).dt.total_seconds() / 86400.0

zm01["falha_proximos_7d"] = (zm01["dias_ate_proxima_falha"] <= 7.0).astype(int)

df_modelo = zm01.dropna(subset=["dias_ate_proxima_falha"]).copy()

In [116]:
colunas_features = [
    "dias_desde_ultima_manutencao",
    "alarmes_ultimas_24h",
    "alarmes_ultimos_7d",
    "tendencia_alarmes",
    "estacao",
    "perfil_do_catalogo",
    "centro_de_trabalho_responsavel",
    "sintoma",
    "objeto",
    "defeito",
    "indisponivel_flag",
    "analisada_flag",
    "ordem_ausente_flag",
    "ordem_repetida_flag",
]

coluna_target = "falha_proximos_7d"

In [117]:
df_modelo = df_modelo.sort_values("T").reset_index(drop=True)

ponto_corte = df_modelo["T"].quantile(0.80)

treino_df = df_modelo[df_modelo["T"] < ponto_corte]
teste_df = df_modelo[df_modelo["T"] >= ponto_corte]

X_train = treino_df[colunas_features]
y_train = treino_df[coluna_target]

X_test = teste_df[colunas_features]
y_test = teste_df[coluna_target]

In [118]:
dados_treino = pd.concat([X_train, y_train], axis=1)
dados_teste = pd.concat([X_test, y_test], axis=1)

# 2. Configura o experimento com Seleção de Features e preserva a divisão temporal
# 2. Configura o PyCaret usando o seu split cronológico exato
exp = setup(
    data=dados_treino,
    test_data=dados_teste,  # Impede o PyCaret de embaralhar as datas
    target=coluna_target,
    feature_selection=True,  # Avalia e remove features irrelevantes/redundantes
    session_id=42,
)

# 3. Avalia e seleciona a melhor arquitetura
melhor_modelo = compare_models(sort="F1")

# 4. Busca os melhores hiperparâmetros via Randomized Search
modelo_otimizado = tune_model(
    melhor_modelo,
    search_library="scikit-learn",
    search_algorithm="random",  # Força RandomizedSearch
    n_iter=25,  # Número de combinações aleatórias testadas
    optimize="F1",
)

# 5. Exibe o gráfico de importância das features selecionadas
plot_model(modelo_otimizado, plot="feature")

NameError: name 'setup' is not defined